# Differential Kinetics

One important concern is dealing with systems that represent multiple lineages and processes, where
genes are likely to show different kinetic regimes across subpopulations. Distinct cell states and
lineages are typically governed by different variations in the gene regulatory networks and may
hence exhibit different splicing kinetics. This gives rise to genes that display multiple trajectories
in phase space. 

To address this, the dynamical model can be used to perform a likelihood-ratio test for differential kinetics. This way, we can detect clusters that display kinetic behavior that cannot be well explained by a single model of the overall dynamics. Clustering cell types into their different kinetic regimes then allows fitting each regime separately.

For illustration, we apply differential kinetic analysis to [dentate gyrus neurogenesis](https://scvelo.readthedocs.io/scvelo.datasets.dentategyrus), which comprises multiple heterogeneous subpopulations.

In [ ]:
# update to the latest version, if not done yet.
# !pip install scvelo --upgrade --quiet

In [ ]:
import scvelo as scv
import pandas as pd
import numpy as np
import matplotlib as plt
import anndata
plt.rcParams['figure.figsize'] = (8, 8)

In [ ]:
scv.settings.verbosity = 3  # show errors(0), warnings(1), info(2), hints(3)
scv.settings.presenter_view = True  # set max width size for presenter view
scv.settings.set_figure_params('scvelo')  # for beautified visualization

### Load and Add metadata and reduction to Anndata
Processing consists of gene selection, log-normalizing, and computing moments. See the previous tutorials for further explanation.

In [ ]:
adata_obs = pd.read_csv("cellID_obs.csv")
umap = pd.read_csv("cell_embeddings_umap.csv")
cell_clusters = pd.read_csv("clusters.csv")
adata = anndata.read_h5ad("../../../../CellRanger/Velocyto.h5ad")
adata

In [ ]:
# Only keep cells from seurat object
adata = adata[np.isin(adata.obs.index,adata_obs["x"])]
adata
# order cells from umap coordinate
adata_index = pd.DataFrame(adata.obs.index)
adata_index = adata_index.rename(columns = {0:'CellID'})
umap = umap.rename(columns = {'Unnamed: 0':'CellID'})
umap_ordered = adata_index.merge(umap,on="CellID")
# Add umap coordinate to Anndata object
umap_ordered = umap_ordered.iloc[:,1:]
adata.obsm['X_umap'] = umap_ordered.values
# reorder cell_cluster
cell_clusters = cell_clusters.rename(columns = {'cells':'CellID'})
clusters_ordered = adata_index.merge(cell_clusters,on="CellID")
# Add cluster info to Anndata object
# adata.obs['clusters'] dtype category must be string, not int
adata.obs['clusters'] = clusters_ordered['RNA_snn_res.0.2'].astype(str).astype('category').values

In [ ]:
scv.pp.filter_and_normalize(adata, min_shared_counts=30, n_top_genes=2000)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

### Basic Velocity Estimation 

In [ ]:
scv.tl.velocity(adata)
scv.tl.velocity_graph(adata, n_jobs = 24)

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='umap')

## Differential Kinetic Test
Distinct cell types and lineages may exhibit different kinetics regimes as these can be governed by a different network structure. Even if cell types or lineages are related, kinetics can be differential due to alternative splicing, alternative polyadenylation and modulations in degradation.

The dynamical model allows us to address this issue with a likelihood ratio test for differential kinetics to detect clusters/lineages that display kinetic behavior that cannot be sufficiently explained by a single model for the overall dynamics. Each cell type is tested whether an independent fit yields a significantly improved likelihood.

The likelihood ratio, following an asymptotic chi-squared distribution, can be tested for significance. Note that for efficiency reasons, by default an orthogonal regression is used instead of a full phase trajectory to test whether a cluster is well explained by the overall kinetic or exhibits a different kinetic.

In [ ]:
# var_names = ['Lin7a', 'Nhs', 'Tenm3', 'Celf2']
# scv.tl.differential_kinetic_test(adata, var_names=var_names, groupby='clusters')

In [ ]:
# scv.get_df(adata[:, var_names], ['fit_diff_kinetics', 'fit_pval_kinetics'], precision=2)

In [ ]:
# kwargs = dict(linewidth=2, add_linfit=True, frameon=False)
# scv.pl.scatter(adata, basis=var_names, add_outline='fit_diff_kinetics', **kwargs)

In *Tmsb10*, for instance, Endothelial display a kinetic behaviour (outlined, with the black line fitted through), that cannot be well explained by the overall dynamics (purple curve).

In [ ]:
# diff_clusters=list(adata[:, var_names].var['fit_diff_kinetics'])
# scv.pl.scatter(adata, legend_loc='right', size=60, title='diff kinetics',
#                add_outline=diff_clusters, outline_width=(.8, .2))

### Testing top-likelihood genes
Screening through the top-likelihood genes, we find some gene-wise dynamics that display multiple kinetic regimes. 

In [ ]:
scv.tl.recover_dynamics(adata, n_jobs = 24)

In [ ]:
top_genes = adata.var['fit_likelihood'].sort_values(ascending=False).index[:100]
scv.tl.differential_kinetic_test(adata, var_names=top_genes, groupby='clusters')
# if run scv.tl.differential_kinetic_test again, it will throw an error:
# ValueError: Data must be 1-dimensional, got ndarray of shape (2000, 1) instead
# which is related to the pd.DataFrame(adata.varm["fit_pvals_kinetics"]).to_numpy() , 
# and adata.varm["fit_pvals_kinetics"] is genereated by previous run

Particularly, cell types that are distinct from the main granule - such as Cck/Tox, GABA, Endothelial, and Microglia - occur frequently. 

In [ ]:
kwargs = dict(linewidth=2, add_linfit=True, frameon=False)
scv.pl.scatter(adata, basis=top_genes[:15], ncols=5, add_outline='fit_diff_kinetics', **kwargs)

In [ ]:
scv.pl.scatter(adata, basis=top_genes[15:30], ncols=5, add_outline='fit_diff_kinetics', **kwargs)

### Recompute velocities

Finally, velocities can be recomputed leveraging the information of multiple competing kinetic regimes.

In [ ]:
scv.tl.velocity(adata, diff_kinetics=True)
scv.tl.velocity_graph(adata)

In [ ]:
scv.pl.velocity_embedding(adata, dpi=120, arrow_size=2, arrow_length=2)

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='umap')